# Pagila: your SQL learning environment

This notebook is a SQL learning environment with the **Pagila** database, a fictional DVD rental business with films, actors, customers, stores, rentals and payments. Use it for the SQL exercises in the *Core SQL and Python for data engineering* module and for the Pagila examples in the *Data normalisation for data integrity* module.

> 📗 **About this browser version:** the original learning environment created a PostgreSQL database on Amazon RDS in an AWS sandbox and connected to it with pgAdmin. Here, the same Pagila tables and rows are in `pagila.sqlite`, a SQLite database in this lab, and you query it in your browser. There is nothing to set up. SQLite's SQL is very close to PostgreSQL's for the `SELECT`, `WHERE`, `GROUP BY` and `JOIN` queries these modules use; the differences that matter are listed at the end of this notebook.

## Setup

Run these cells first, each time you open the notebook. They install `jupysql`, which runs the SQL in any cell that starts with `%%sql`, and load a fresh copy of Pagila into memory. Because each session starts from a fresh copy, nothing you change is permanent: run the setup cells again to start over.

In [ ]:
# Hide some harmless but distracting warnings from jupysql
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

In [ ]:
%pip install -q jupysql
%load_ext sql

In [ ]:
import sqlite3
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool

# Copy pagila.sqlite into an in-memory database, so every session starts with the original data
source = sqlite3.connect('pagila.sqlite')
con = sqlite3.connect(':memory:', check_same_thread=False)
source.backup(con)
source.close()

# Connect jupysql to that one connection
engine = create_engine('sqlite://', creator=lambda: con, poolclass=StaticPool)
%sql engine

## The tables

In pgAdmin you would expand *Schemas > public > Tables* to see the tables. In SQLite, every table and view is listed in the `sqlite_master` table:

In [ ]:
%%sql
SELECT type, name
FROM sqlite_master
WHERE type IN ('table', 'view')
  AND name NOT LIKE 'sqlite_%'
ORDER BY type, name;

jupysql shows the first 10 rows of a result. To see them all, run `%config SqlMagic.displaylimit = None` in a code cell.

To see a table's columns and their types, use `pragma_table_info`. Change `'film'` to look at another table:

In [ ]:
%%sql
SELECT name, type, "notnull", pk
FROM pragma_table_info('film');

## How the tables relate

These are the relationships in Pagila: each row says which column links two tables and how many rows on each side can match. "1 : many" means one row in the first table can link to many rows in the second.

| Table | Related table | Linked by | Relationship |
| --- | --- | --- | --- |
| country | city | `city.country_id` | 1 : many (each city is in exactly one country) |
| city | address | `address.city_id` | 1 : many |
| address | customer, staff, store | `address_id` | 1 : many |
| store | customer, staff, inventory | `store_id` | 1 : many |
| language | film | `film.language_id` | 1 : many |
| film | inventory | `inventory.film_id` | 1 : many (one film, many copies) |
| actor | film | `film_actor` (`actor_id`, `film_id`) | many : many |
| film | category | `film_category` (`film_id`, `category_id`) | many : many |
| inventory | rental | `rental.inventory_id` | 1 : many |
| customer | rental | `rental.customer_id` | 1 : many |
| staff | rental | `rental.staff_id` | 1 : many |
| rental | payment | `payment.rental_id` | 1 : many |
| customer | payment | `payment.customer_id` | 1 : many (a customer makes payments) |
| staff | payment | `payment.staff_id` | 1 : many |

`film_actor` and `film_category` are junction tables: they turn each many-to-many relationship into two one-to-many relationships. `store.manager_staff_id` also points at a member of staff, but Pagila does not declare it as a foreign key.

The database itself records each foreign key. This query lists them all:

In [ ]:
%%sql
SELECT m.name AS "table", fk."from" AS "column", fk."table" AS references_table, fk."to" AS references_column
FROM sqlite_master AS m
JOIN pragma_foreign_key_list(m.name) AS fk
WHERE m.type = 'table'
ORDER BY m.name;

## Sample queries: SELECT, LIMIT, DISTINCT and WHERE

The first five films, with every column:

In [ ]:
%%sql
SELECT *
FROM film
LIMIT 5;

The different ratings films can have, each listed once:

In [ ]:
%%sql
SELECT DISTINCT rating
FROM film;

Films rated PG that last longer than two hours:

In [ ]:
%%sql
SELECT title, length, rating
FROM film
WHERE rating = 'PG'
  AND length > 120
ORDER BY length DESC;

## Sample queries: GROUP BY, COUNT, SUM, MIN, MAX and ORDER BY

How many films there are for each rating, with the shortest and longest:

In [ ]:
%%sql
SELECT rating,
       COUNT(*) AS films,
       MIN(length) AS shortest,
       MAX(length) AS longest
FROM film
GROUP BY rating
ORDER BY films DESC;

The five customers who have paid the most. SQLite stores `amount` as a floating-point number, so `ROUND` keeps the totals to the penny:

In [ ]:
%%sql
SELECT customer_id,
       COUNT(*) AS payments,
       ROUND(SUM(amount), 2) AS total_paid
FROM payment
GROUP BY customer_id
ORDER BY total_paid DESC
LIMIT 5;

## Sample queries: INNER JOIN and LEFT JOIN

Each city with its country. An `INNER JOIN` returns only the rows that match in both tables:

In [ ]:
%%sql
SELECT city.city, country.country
FROM city
INNER JOIN country ON city.country_id = country.country_id
ORDER BY country.country, city.city
LIMIT 10;

The one-to-many relationship between country and city, counted: the countries with the most cities.

In [ ]:
%%sql
SELECT country.country, COUNT(city.city_id) AS cities
FROM country
INNER JOIN city ON city.country_id = country.country_id
GROUP BY country.country
ORDER BY cities DESC
LIMIT 5;

Films that no store has a copy of. A `LEFT JOIN` keeps every film, with empty (NULL) inventory columns where there is no match, so `WHERE i.inventory_id IS NULL` finds the films with no copies:

In [ ]:
%%sql
SELECT f.film_id, f.title
FROM film AS f
LEFT JOIN inventory AS i ON f.film_id = i.film_id
WHERE i.inventory_id IS NULL;

## Sample queries: the actor, film and category entities

The *Data normalisation* module builds a conceptual model of three Pagila entities: an **actor** *is in* a **film**, and a **film** *is in* a **category**. Both relationships are many-to-many, held in the junction tables `film_actor` and `film_category`.

The inventory table has three keys: `inventory_id` is its primary key, and `film_id` and `store_id` are foreign keys to the film and store tables:

In [ ]:
%%sql
SELECT inventory_id, film_id, store_id
FROM inventory
LIMIT 5;

Actors, the films they are in, and each film's category, with the film attributes from the conceptual model (title, length and description):

In [ ]:
%%sql
SELECT a.first_name, a.last_name, f.title, f.length, f.description, c.name AS category
FROM actor AS a
INNER JOIN film_actor AS fa ON a.actor_id = fa.actor_id
INNER JOIN film AS f ON fa.film_id = f.film_id
INNER JOIN film_category AS fc ON f.film_id = fc.film_id
INNER JOIN category AS c ON fc.category_id = c.category_id
ORDER BY a.last_name, a.first_name, f.title
LIMIT 10;

Many-to-many in both directions: how many films each actor is in, and how many actors each film has. The second query uses a `LEFT JOIN`, so it also finds films with no actors at all: the relationship is optional.

In [ ]:
%%sql
SELECT a.first_name, a.last_name, COUNT(fa.film_id) AS films
FROM actor AS a
INNER JOIN film_actor AS fa ON a.actor_id = fa.actor_id
GROUP BY a.actor_id, a.first_name, a.last_name
ORDER BY films DESC
LIMIT 5;

In [ ]:
%%sql
SELECT f.title, COUNT(fa.actor_id) AS actors
FROM film AS f
LEFT JOIN film_actor AS fa ON f.film_id = fa.film_id
GROUP BY f.film_id, f.title
ORDER BY actors ASC
LIMIT 5;

## How this SQLite copy differs from PostgreSQL

`generate_db.py` in this lab builds `pagila.sqlite` from the same Pagila files the AWS environment loaded. The tables, columns and rows are the same. These are the differences you might notice:

- **Data types:** SQLite has fewer types. Booleans (`customer.activebool`, `staff.active`) are stored as 1 and 0, the film `rating` and `release_year` types are text and integer columns with `CHECK` constraints, and amounts are stored as floating-point numbers.
- **Arrays and full-text search:** `film.special_features` holds PostgreSQL's array text, such as `{Trailers,"Deleted Scenes"}`. Search it with `LIKE '%Trailers%'`. `film.fulltext` holds the text of PostgreSQL's search vector, but SQLite cannot search it the way PostgreSQL can.
- **Payments:** PostgreSQL splits `payment` into monthly partitions (`payment_p2022_01` to `payment_p2022_07`). Here it is one `payment` table.
- **Functions, triggers and materialized views:** Pagila's PostgreSQL functions (such as `film_in_stock` and `get_customer_balance`) and its triggers do not exist in SQLite, and `rental_by_category` is an ordinary view.
- **Dates:** timestamps are stored as text with their time-zone offset, such as `2022-01-23 13:03:52.212496+00`. Use SQLite's date functions, such as `strftime('%Y-%m', payment_date)`, in place of PostgreSQL's `date_trunc` and `extract`.
- **SQL dialect:** a few PostgreSQL features differ, for example `ILIKE` (in SQLite, `LIKE` already ignores case for English letters) and `::` casts (use `CAST(x AS INTEGER)`).

> 📗 SQLite's SQL is documented at https://www.sqlite.org/lang.html